# Clase 3 — De los datos a un servicio de inferencia

Caso practico: estimar el **precio de un vehiculo usado** (regresion).

Enfoque: como preparar datos, entrenar un baseline, evaluarlo con metricas
interpretables y dejarlo listo para un servicio de inferencia (API).

In [2]:
import pandas as pd

df = pd.read_csv('/content/CAR DETAILS FROM CAR DEKHO.csv')

print("Dimensiones:", df.shape)
print("\nColumnas:")
print(df.columns.tolist())

print("\nPrimeras filas:")
display(df.head())

Dimensiones: (4340, 8)

Columnas:
['name', 'year', 'selling_price', 'km_driven', 'fuel', 'seller_type', 'transmission', 'owner']

Primeras filas:


,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner
0,Maruti 800 AC,2007,60000,70000,Petrol,Individual,Manual,First Owner
1,Maruti Wagon R LXI Minor,2007,135000,50000,Petrol,Individual,Manual,First Owner
2,Hyundai Verna 1.6 SX,2012,600000,100000,Diesel,Individual,Manual,First Owner
3,Datsun RediGO T Option,2017,250000,46000,Petrol,Individual,Manual,First Owner
4,Honda Amaze VX i-DTEC,2014,450000,141000,Diesel,Individual,Manual,Second Owner


## 1) ¿Regresion o clasificacion?

La primera decision no es que biblioteca usar, sino **que queremos predecir**.

| Pregunta | Tipo | Ejemplo |
|---|---|---|
| ¿Cuanto costara? | Regresion | Precio de un automovil |
| ¿Ocurrira? | Clasificacion | Contratara: Si/No |
| ¿A que grupo pertenece? | Clasificacion | Tipo de cliente |

En esta clase seguimos un problema completo de **regresion** (precio de
vehiculos) de principio a fin, y al final contrastamos brevemente Regresion
Logistica y KNN para ver como cambia el enfoque cuando el target ya no es
numerico.

In [3]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option('display.max_columns', None)

## 2) Recibo un dataset: revision en 5 minutos

Antes de modelar necesitamos responder rapido:

- ¿cuantos datos tengo? (`shape`)
- ¿que significa cada variable? (`dtypes`)
- ¿que esta incompleto? (`isna().sum()`)
- ¿que parece incorrecto? (`duplicated().sum()`, `describe()`)
- ¿cual es mi target? → **`precio`**

Y antes de seguir, verificar unidades: `precio` en MXN, `km` en kilometros.
Una inconsistencia de unidades puede destruir un modelo.

In [4]:
MODELOS_POR_MARCA = {
    'Nissan': ['Sentra', 'Versa'],
    'Mazda': ['CX-5', 'Mazda 3'],
    'Volkswagen': ['Jetta', 'Vento'],
    'Toyota': ['Corolla', 'Yaris'],
    'Chevrolet': ['Aveo', 'Onix'],
}

PRECIO_BASE_MARCA = {
    'Nissan': 260000,
    'Mazda': 300000,
    'Volkswagen': 270000,
    'Toyota': 290000,
    'Chevrolet': 230000,
}


def build_synthetic_dataset(seed: int = 42, n_rows: int = 1500) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    marcas = list(PRECIO_BASE_MARCA.keys())

    rows = []
    for _ in range(n_rows):
        marca = rng.choice(marcas)
        modelo = rng.choice(MODELOS_POR_MARCA[marca])
        anio = int(rng.integers(2015, 2024))
        km = int(max(1000, rng.normal((2024 - anio) * 12000, 9000)))
        transmision = rng.choice(['Automatica', 'Manual'], p=[0.65, 0.35])

        precio = (
            PRECIO_BASE_MARCA[marca]
            + (anio - 2015) * 18000
            - km * 1.1
            + (5000 if transmision == 'Automatica' else 0)
            # ruido amplio: representa factores no capturados (estado real, mantenimiento)
            + rng.normal(0, 30000)
        )
        precio = float(np.clip(precio, 60000, None))

        rows.append({
            'marca': marca,
            'modelo': modelo,
            'anio': anio,
            'km': km,
            'transmision': transmision,
            'precio': round(precio, 2),
        })

    return pd.DataFrame(rows)


df = build_synthetic_dataset(seed=42, n_rows=1500)
print('shape:', df.shape)
df.head()

shape: (1500, 6)


,marca,modelo,anio,km,transmision,precio
0,Nissan,Versa,2020,54754,Manual,231239.54
1,Volkswagen,Vento,2023,13150,Manual,399030.97
2,Chevrolet,Aveo,2019,67000,Automatica,267117.24
3,Mazda,Mazda 3,2018,64266,Automatica,259540.92
4,Chevrolet,Onix,2017,82336,Automatica,217106.64


In [5]:
print('dtypes:')
print(df.dtypes)
print()
print('valores nulos:')
print(df.isna().sum())
print()
print('duplicados:', df.duplicated().sum())

dtypes:
marca           object
modelo          object
anio             int64
km               int64
transmision     object
precio         float64
dtype: object

valores nulos:
marca          0
modelo         0
anio           0
km             0
transmision    0
precio         0
dtype: int64

duplicados: 0


In [6]:
df.describe()

,anio,km,precio
count,1500.0000,1500.000000,1500.000000
mean,2019.0700,59037.375333,281136.916533
std,2.6375,32316.215441,90780.142861
min,2015.0000,1000.000000,60000.000000
25%,2017.0000,30901.750000,211505.057500
50%,2019.0000,57407.500000,283898.760000
75%,2021.0000,86877.750000,356241.142500
max,2023.0000,130434.000000,509850.730000


In [7]:
print('marca:')
print(df['marca'].value_counts())
print()
print('transmision:')
print(df['transmision'].value_counts())

marca:
marca
Volkswagen    317
Toyota        297
Mazda         296
Nissan        295
Chevrolet     295
Name: count, dtype: int64

transmision:
transmision
Automatica    1027
Manual         473
Name: count, dtype: int64


CARGAR DATASET de Kaggle y no la Synthetic

In [8]:
# Cargar dataset real de Kaggle

df = pd.read_csv('/content/CAR DETAILS FROM CAR DEKHO.csv')

print('shape:', df.shape)
display(df.head())

shape: (4340, 8)


,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner
0,Maruti 800 AC,2007,60000,70000,Petrol,Individual,Manual,First Owner
1,Maruti Wagon R LXI Minor,2007,135000,50000,Petrol,Individual,Manual,First Owner
2,Hyundai Verna 1.6 SX,2012,600000,100000,Diesel,Individual,Manual,First Owner
3,Datsun RediGO T Option,2017,250000,46000,Petrol,Individual,Manual,First Owner
4,Honda Amaze VX i-DTEC,2014,450000,141000,Diesel,Individual,Manual,Second Owner


Mejorar Datos

In [9]:
# Renombrar columnas para trabajar con nombres más claros

df = df.rename(columns={
    'year': 'anio',
    'selling_price': 'precio',
    'km_driven': 'km',
    'transmission': 'transmision'
})

print(df.columns.tolist())
display(df.head())

['name', 'anio', 'precio', 'km', 'fuel', 'seller_type', 'transmision', 'owner']


,name,anio,precio,km,fuel,seller_type,transmision,owner
0,Maruti 800 AC,2007,60000,70000,Petrol,Individual,Manual,First Owner
1,Maruti Wagon R LXI Minor,2007,135000,50000,Petrol,Individual,Manual,First Owner
2,Hyundai Verna 1.6 SX,2012,600000,100000,Diesel,Individual,Manual,First Owner
3,Datsun RediGO T Option,2017,250000,46000,Petrol,Individual,Manual,First Owner
4,Honda Amaze VX i-DTEC,2014,450000,141000,Diesel,Individual,Manual,Second Owner


## 3) Estadistica util antes de modelar

No necesitamos calcular todo. Necesitamos detectar comportamiento relevante:

- **Tendencia central**: media vs. mediana de `precio`.
- **Dispersion**: desviacion estandar / IQR de `km`.
- **Posicion**: percentiles.
- **Relacion**: correlacion `anio` ↔ `precio` (esperada positiva) y
  `km` ↔ `precio` (esperada negativa).

Un vehiculo con precio muy alto no se elimina automaticamente: primero
preguntamos si es un error o un vehiculo de alta gama. **Atipico ≠ incorrecto.**

In [10]:
# 3. Estadística antes de modelar

print('--- Estadística del precio ---')
print('Precio -> media:', round(df['precio'].mean(), 2))
print('Precio -> mediana:', round(df['precio'].median(), 2))
print('Precio -> desviación estándar:', round(df['precio'].std(), 2))

print()

print('--- Estadística de kilómetros ---')
q1, q3 = df['km'].quantile([0.25, 0.75])
print('Km -> IQR:', round(q3 - q1, 2))
print('Km -> percentil 90:', round(df['km'].quantile(0.9), 2))

print()

print('--- Valores faltantes ---')
print(df.isnull().sum())

print()

print('--- Correlación con precio ---')
print(df[['anio', 'km', 'precio']].corr()['precio'])

print()

print('--- Variables categóricas ---')
print('Combustible:', df['fuel'].unique())
print('Tipo de vendedor:', df['seller_type'].unique())
print('Transmisión:', df['transmision'].unique())
print('Propietarios:', df['owner'].unique())

--- Estadística del precio ---
Precio -> media: 504127.31
Precio -> mediana: 350000.0
Precio -> desviación estándar: 578548.74

--- Estadística de kilómetros ---
Km -> IQR: 55000.0
Km -> percentil 90: 120000.0

--- Valores faltantes ---
name           0
anio           0
precio         0
km             0
fuel           0
seller_type    0
transmision    0
owner          0
dtype: int64

--- Correlación con precio ---
anio      0.413922
km       -0.192289
precio    1.000000
Name: precio, dtype: float64

--- Variables categóricas ---
Combustible: ['Petrol' 'Diesel' 'CNG' 'LPG' 'Electric']
Tipo de vendedor: ['Individual' 'Dealer' 'Trustmark Dealer']
Transmisión: ['Manual' 'Automatic']
Propietarios: ['First Owner' 'Second Owner' 'Fourth & Above Owner' 'Third Owner'
 'Test Drive Car']


## 4) Preparar los datos que realmente necesita el modelo

Separamos:

- **X** (predictoras): `marca`, `modelo`, `anio`, `km`, `transmision`
- **y** (target): `precio`

Despues: `train` 80% / `test` 20%.

Y por tipo de variable:

- **Categoricas** (`marca`, `modelo`, `transmision`) → encoding (One-Hot).
- **Numericas** (`anio`, `km`) → se usan directo (Regresion Lineal no las
  necesita escaladas, pero mas adelante veremos que **KNN si**).

Dos reglas importantes:

1. El test no debe usarse para aprender transformaciones.
2. La transformacion usada en entrenamiento debe ser exactamente la misma en
   inferencia.

Por eso encapsulamos **preprocessing + modelo** en un `Pipeline`.

In [11]:
# 4. Preparar los datos que realmente necesita el modelo

CATEGORICAL_FEATURES = [
    'fuel',
    'seller_type',
    'transmision',
    'owner'
]

NUMERIC_FEATURES = [
    'anio',
    'km'
]

FEATURE_ORDER = CATEGORICAL_FEATURES + NUMERIC_FEATURES

TARGET = 'precio'

X = df[FEATURE_ORDER]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            'categorical',
            OneHotEncoder(handle_unknown='ignore'),
            CATEGORICAL_FEATURES
        ),
    ],
    remainder='passthrough'
)

print('train:', X_train.shape, '| test:', X_test.shape)

train: (3472, 6) | test: (868, 6)


## 5) Regresion Lineal: nuestro baseline

La Regresion Lineal intenta aproximar una relacion entre las caracteristicas
y una variable numerica (`precio`).

¿Por que empezar aqui?

- sencilla
- rapida
- interpretable
- **util como baseline**

No afirmamos que sea el mejor modelo para valuacion vehicular. Establecemos
una referencia: si mas adelante un modelo mas complejo da un MAE menor,
sabremos cuanto realmente mejora.

In [12]:
# 5. Regresión Lineal: nuestro baseline

price_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression()),
])

price_model.fit(X_train, y_train)

ejemplo = pd.DataFrame([{
    'fuel': 'Petrol',
    'seller_type': 'Individual',
    'transmision': 'Manual',
    'owner': 'First Owner',
    'anio': 2018,
    'km': 50000,
}])

print('Precio estimado (ejemplo):', round(float(price_model.predict(ejemplo)[0]), 2))

Precio estimado (ejemplo): 445566.9


## 6) ¿Como se si mi regresion funciona?

- **MAE** (error absoluto medio): facil de comunicar, en las mismas
  unidades que el target (pesos).
- **RMSE**: penaliza mas los errores grandes. Si es mucho mayor que el MAE,
  hay algunos errores especialmente grandes que vale la pena investigar.
- **R²**: que proporcion de la variabilidad del precio explica el modelo.

No preguntar solo "¿R² es alto?". Preguntar: **¿el error es aceptable para
el uso que le quiero dar?** Un MAE de $24,000 puede ser aceptable para un
vehiculo de $1.5M y problematico para uno de $150,000.

In [13]:
pred = price_model.predict(X_test)

mae = mean_absolute_error(y_test, pred)
rmse = mean_squared_error(y_test, pred) ** 0.5
r2 = r2_score(y_test, pred)

print(f'MAE:  ${mae:,.2f} MXN')
print(f'RMSE: ${rmse:,.2f} MXN')
print(f'R2:   {r2:.4f}')

MAE:  $221,706.37 MXN
RMSE: $426,786.69 MXN
R2:   0.4031


Experimento 1 — Creación de la variable antigüedad

Objetivo: mejorar la capacidad predictiva del modelo incorporando una nueva variable derivada del año del vehículo.

La variable anio indica el año en que fue fabricado el vehículo, pero podemos transformarla en una medida más interpretable: su antigüedad.

Para ello, se crea la variable:

antiguedad = 2026 - anio

La idea es que la antigüedad puede representar de manera más directa el desgaste y la depreciación del vehículo. Se comparará el nuevo modelo contra el Baseline, manteniendo el mismo algoritmo de Regresión Lineal y el mismo conjunto de entrenamiento y prueba.

In [14]:
# Experimento 1: crear variable antigüedad

df_exp1 = df.copy()

df_exp1['antiguedad'] = 2026 - df_exp1['anio']

display(df_exp1[['anio', 'antiguedad']].head())

,anio,antiguedad
0,2007,19
1,2007,19
2,2012,14
3,2017,9
4,2014,12


In [15]:
# Preparar datos para el Experimento 1

CATEGORICAL_FEATURES_EXP1 = [
    'fuel',
    'seller_type',
    'transmision',
    'owner'
]

NUMERIC_FEATURES_EXP1 = [
    'anio',
    'km',
    'antiguedad'
]

FEATURE_ORDER_EXP1 = CATEGORICAL_FEATURES_EXP1 + NUMERIC_FEATURES_EXP1

X_exp1 = df_exp1[FEATURE_ORDER_EXP1]
y_exp1 = df_exp1['precio']

X_train_exp1, X_test_exp1, y_train_exp1, y_test_exp1 = train_test_split(
    X_exp1,
    y_exp1,
    test_size=0.2,
    random_state=42
)

preprocessor_exp1 = ColumnTransformer(
    transformers=[
        (
            'categorical',
            OneHotEncoder(handle_unknown='ignore'),
            CATEGORICAL_FEATURES_EXP1
        ),
    ],
    remainder='passthrough'
)

print('train:', X_train_exp1.shape, '| test:', X_test_exp1.shape)

train: (3472, 7) | test: (868, 7)


In [16]:
# Entrenar modelo del Experimento 1

price_model_exp1 = Pipeline(steps=[
    ('preprocessor', preprocessor_exp1),
    ('regressor', LinearRegression()),
])

price_model_exp1.fit(X_train_exp1, y_train_exp1)

pred_exp1 = price_model_exp1.predict(X_test_exp1)

mae_exp1 = mean_absolute_error(y_test_exp1, pred_exp1)
rmse_exp1 = mean_squared_error(y_test_exp1, pred_exp1) ** 0.5
r2_exp1 = r2_score(y_test_exp1, pred_exp1)

print(f'MAE Experimento 1:  ${mae_exp1:,.2f} MXN')
print(f'RMSE Experimento 1: ${rmse_exp1:,.2f} MXN')
print(f'R² Experimento 1:   {r2_exp1:.4f}')

MAE Experimento 1:  $221,706.37 MXN
RMSE Experimento 1: $426,786.69 MXN
R² Experimento 1:   0.4031


Resultados del Experimento 1

La incorporación de la variable antiguedad no produjo cambios en las métricas del modelo. El MAE se mantuvo en 221,706.37, el RMSE en 426,786.69 y el R² en 0.4031, exactamente igual que en el Baseline.

Esto se debe a que antiguedad se obtiene directamente a partir de anio mediante una transformación lineal (2026 - anio). Por lo tanto, no aporta información adicional al modelo de Regresión Lineal que no estuviera disponible previamente.

Conclusión: el experimento no mejoró ni empeoró el desempeño del modelo, por lo que el Baseline continúa siendo superior por simplicidad.

Experimento 2 — Tratamiento de outliers

Los valores atípicos pueden afectar el desempeño de una Regresión Lineal, especialmente cuando existen precios o kilometrajes muy alejados del comportamiento general de los datos.

En este experimento se identificarán valores extremos utilizando el rango intercuartílico (IQR). Se aplicará el tratamiento sobre las variables numéricas precio y km, eliminando los registros que se encuentren fuera de los límites definidos por el IQR.

Posteriormente, se entrenará nuevamente el modelo de Regresión Lineal y se compararán sus métricas con el Baseline.

In [17]:
# Experimento 2: identificación y tratamiento de outliers

df_exp2 = df.copy()

def limites_iqr(data, columna):
    q1 = data[columna].quantile(0.25)
    q3 = data[columna].quantile(0.75)
    iqr = q3 - q1

    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr

    return limite_inferior, limite_superior

precio_inf, precio_sup = limites_iqr(df_exp2, 'precio')
km_inf, km_sup = limites_iqr(df_exp2, 'km')

print('Precio -> límite inferior:', round(precio_inf, 2))
print('Precio -> límite superior:', round(precio_sup, 2))

print()

print('Km -> límite inferior:', round(km_inf, 2))
print('Km -> límite superior:', round(km_sup, 2))

print()

print('Registros originales:', len(df_exp2))

df_exp2 = df_exp2[
    (df_exp2['precio'] >= precio_inf) &
    (df_exp2['precio'] <= precio_sup) &
    (df_exp2['km'] >= km_inf) &
    (df_exp2['km'] <= km_sup)
]

print('Registros después de eliminar outliers:', len(df_exp2))
print('Outliers eliminados:', len(df) - len(df_exp2))

Precio -> límite inferior: -378125.62
Precio -> límite superior: 1186875.38

Km -> límite inferior: -47500.0
Km -> límite superior: 172500.0

Registros originales: 4340
Registros después de eliminar outliers: 3962
Outliers eliminados: 378


In [18]:
# Preparar datos para el Experimento 2

CATEGORICAL_FEATURES_EXP2 = [
    'fuel',
    'seller_type',
    'transmision',
    'owner'
]

NUMERIC_FEATURES_EXP2 = [
    'anio',
    'km'
]

FEATURE_ORDER_EXP2 = CATEGORICAL_FEATURES_EXP2 + NUMERIC_FEATURES_EXP2

X_exp2 = df_exp2[FEATURE_ORDER_EXP2]
y_exp2 = df_exp2['precio']

X_train_exp2, X_test_exp2, y_train_exp2, y_test_exp2 = train_test_split(
    X_exp2,
    y_exp2,
    test_size=0.2,
    random_state=42
)

preprocessor_exp2 = ColumnTransformer(
    transformers=[
        (
            'categorical',
            OneHotEncoder(handle_unknown='ignore'),
            CATEGORICAL_FEATURES_EXP2
        ),
    ],
    remainder='passthrough'
)

print('train:', X_train_exp2.shape, '| test:', X_test_exp2.shape)

train: (3169, 6) | test: (793, 6)


In [19]:
# Entrenar y evaluar el Experimento 2

price_model_exp2 = Pipeline(steps=[
    ('preprocessor', preprocessor_exp2),
    ('regressor', LinearRegression()),
])

price_model_exp2.fit(X_train_exp2, y_train_exp2)

pred_exp2 = price_model_exp2.predict(X_test_exp2)

mae_exp2 = mean_absolute_error(y_test_exp2, pred_exp2)
rmse_exp2 = mean_squared_error(y_test_exp2, pred_exp2) ** 0.5
r2_exp2 = r2_score(y_test_exp2, pred_exp2)

print(f'MAE Experimento 2:  ${mae_exp2:,.2f} MXN')
print(f'RMSE Experimento 2: ${rmse_exp2:,.2f} MXN')
print(f'R² Experimento 2:   {r2_exp2:.4f}')

MAE Experimento 2:  $125,692.75 MXN
RMSE Experimento 2: $168,867.19 MXN
R² Experimento 2:   0.5445


Resultados del Experimento 2

El tratamiento de valores atípicos produjo una mejora considerable en el desempeño de la Regresión Lineal. El MAE disminuyó de 221,706.37 a 125,692.75 MXN, mientras que el RMSE pasó de 426,786.69 a 168,867.19 MXN. Por otro lado, el R² aumentó de 0.4031 a 0.5445.

La mejora puede explicarse porque los valores extremos de precio y kilometraje tenían una influencia importante sobre la Regresión Lineal. Al eliminar 378 registros considerados outliers mediante el criterio del IQR, el modelo pudo ajustarse mejor al comportamiento general de los vehículos.

Conclusión: el tratamiento de outliers mejoró significativamente el desempeño del modelo. De los experimentos realizados hasta este punto, este es el que ha producido la mayor mejora.

Experimento 3 — Incorporación de la variable name

La variable name contiene información sobre la marca, modelo y versión del vehículo. En los experimentos anteriores esta variable no fue utilizada, por lo que se evaluará si incorporarla mejora la capacidad predictiva del modelo.

Para realizar una comparación justa, se mantendrá el tratamiento de outliers utilizado en el Experimento 2 y se agregará name como variable categórica. De esta manera, el único cambio respecto al Experimento 2 será la incorporación de esta nueva característica.

In [20]:
# Experimento 3: incorporar la variable name

CATEGORICAL_FEATURES_EXP3 = [
    'name',
    'fuel',
    'seller_type',
    'transmision',
    'owner'
]

NUMERIC_FEATURES_EXP3 = [
    'anio',
    'km'
]

FEATURE_ORDER_EXP3 = CATEGORICAL_FEATURES_EXP3 + NUMERIC_FEATURES_EXP3

X_exp3 = df_exp2[FEATURE_ORDER_EXP3]
y_exp3 = df_exp2['precio']

X_train_exp3, X_test_exp3, y_train_exp3, y_test_exp3 = train_test_split(
    X_exp3,
    y_exp3,
    test_size=0.2,
    random_state=42
)

preprocessor_exp3 = ColumnTransformer(
    transformers=[
        (
            'categorical',
            OneHotEncoder(handle_unknown='ignore'),
            CATEGORICAL_FEATURES_EXP3
        ),
    ],
    remainder='passthrough'
)

print('train:', X_train_exp3.shape, '| test:', X_test_exp3.shape)

train: (3169, 7) | test: (793, 7)


In [21]:
# Entrenar y evaluar el Experimento 3

price_model_exp3 = Pipeline(steps=[
    ('preprocessor', preprocessor_exp3),
    ('regressor', LinearRegression()),
])

price_model_exp3.fit(X_train_exp3, y_train_exp3)

pred_exp3 = price_model_exp3.predict(X_test_exp3)

mae_exp3 = mean_absolute_error(y_test_exp3, pred_exp3)
rmse_exp3 = mean_squared_error(y_test_exp3, pred_exp3) ** 0.5
r2_exp3 = r2_score(y_test_exp3, pred_exp3)

print(f'MAE Experimento 3:  ${mae_exp3:,.2f} MXN')
print(f'RMSE Experimento 3: ${rmse_exp3:,.2f} MXN')
print(f'R² Experimento 3:   {r2_exp3:.4f}')

MAE Experimento 3:  $128,555.80 MXN
RMSE Experimento 3: $171,485.54 MXN
R² Experimento 3:   0.5302


Resultados del Experimento 3

La incorporación de la variable name no mejoró el desempeño del modelo. El MAE aumentó de 125,692.75 a 128,555.80 MXN, mientras que el RMSE aumentó de 168,867.19 a 171,485.54 MXN. Además, el R² disminuyó de 0.5445 a 0.5302.

Una posible explicación es que la variable name contiene una gran cantidad de categorías diferentes. Al aplicar One-Hot Encoding se generan numerosas variables, lo que puede dificultar la generalización del modelo y no necesariamente aporta información adicional suficiente para mejorar la predicción.

Conclusión: incorporar directamente name como variable categórica empeoró ligeramente el desempeño, por lo que el modelo del Experimento 2 continúa siendo superior.

Experimento 4 — Cambio de algoritmo a Random Forest

En los experimentos anteriores se utilizó Regresión Lineal, que supone una relación aproximadamente lineal entre las variables predictoras y el precio.

En este experimento se utilizará Random Forest Regressor, un algoritmo basado en múltiples árboles de decisión que puede capturar relaciones no lineales e interacciones entre las características del vehículo.

Se mantendrá el tratamiento de outliers del Experimento 2 y las mismas variables utilizadas en dicho experimento. De esta manera, se evaluará específicamente si el cambio de algoritmo mejora el desempeño.

In [22]:
from sklearn.ensemble import RandomForestRegressor

In [23]:
# Experimento 4: Random Forest

CATEGORICAL_FEATURES_EXP4 = [
    'fuel',
    'seller_type',
    'transmision',
    'owner'
]

NUMERIC_FEATURES_EXP4 = [
    'anio',
    'km'
]

FEATURE_ORDER_EXP4 = CATEGORICAL_FEATURES_EXP4 + NUMERIC_FEATURES_EXP4

X_exp4 = df_exp2[FEATURE_ORDER_EXP4]
y_exp4 = df_exp2['precio']

X_train_exp4, X_test_exp4, y_train_exp4, y_test_exp4 = train_test_split(
    X_exp4,
    y_exp4,
    test_size=0.2,
    random_state=42
)

preprocessor_exp4 = ColumnTransformer(
    transformers=[
        (
            'categorical',
            OneHotEncoder(handle_unknown='ignore'),
            CATEGORICAL_FEATURES_EXP4
        ),
    ],
    remainder='passthrough'
)

print('train:', X_train_exp4.shape, '| test:', X_test_exp4.shape)

train: (3169, 6) | test: (793, 6)


In [24]:
# Entrenar y evaluar el Experimento 4

price_model_exp4 = Pipeline(steps=[
    ('preprocessor', preprocessor_exp4),
    ('regressor', RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    )),
])

price_model_exp4.fit(X_train_exp4, y_train_exp4)

pred_exp4 = price_model_exp4.predict(X_test_exp4)

mae_exp4 = mean_absolute_error(y_test_exp4, pred_exp4)
rmse_exp4 = mean_squared_error(y_test_exp4, pred_exp4) ** 0.5
r2_exp4 = r2_score(y_test_exp4, pred_exp4)

print(f'MAE Experimento 4:  ${mae_exp4:,.2f} MXN')
print(f'RMSE Experimento 4: ${rmse_exp4:,.2f} MXN')
print(f'R² Experimento 4:   {r2_exp4:.4f}')

MAE Experimento 4:  $108,486.47 MXN
RMSE Experimento 4: $158,551.03 MXN
R² Experimento 4:   0.5984


Resultados del Experimento 4

El cambio de Regresión Lineal a Random Forest produjo una mejora en las tres métricas evaluadas. El MAE disminuyó de 125,692.75 a 108,486.47 MXN, mientras que el RMSE pasó de 168,867.19 a 158,551.03 MXN. El R² también aumentó de 0.5445 a 0.5984.

La mejora puede explicarse porque Random Forest puede capturar relaciones no lineales e interacciones entre las características de los vehículos que la Regresión Lineal no puede representar adecuadamente.

Conclusión: Random Forest superó a la Regresión Lineal y se convierte en el mejor modelo hasta este punto.

Experimento 5 — Ajuste de hiperparámetros de Random Forest

Después de seleccionar Random Forest como el mejor algoritmo hasta este punto, se realizará un ajuste de sus hiperparámetros para buscar un mejor desempeño.

Se probarán diferentes valores de n_estimators, max_depth, min_samples_split y min_samples_leaf. El objetivo es encontrar una configuración que permita mejorar las métricas obtenidas en el Experimento 4 sin cambiar el conjunto de variables ni el tratamiento de outliers.

In [25]:
from sklearn.model_selection import GridSearchCV

In [26]:
# Experimento 5: búsqueda de hiperparámetros

rf_base = RandomForestRegressor(
    random_state=42,
    n_jobs=-1
)

price_model_exp5 = Pipeline(steps=[
    ('preprocessor', preprocessor_exp4),
    ('regressor', rf_base)
])

param_grid = {
    'regressor__n_estimators': [100, 200],
    'regressor__max_depth': [None, 10, 20],
    'regressor__min_samples_split': [2, 5],
    'regressor__min_samples_leaf': [1, 2]
}

grid_search = GridSearchCV(
    estimator=price_model_exp5,
    param_grid=param_grid,
    cv=3,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)

grid_search.fit(X_train_exp4, y_train_exp4)

print('Mejores parámetros:')
print(grid_search.best_params_)

print()
print('Mejor RMSE de validación cruzada:',
      round(-grid_search.best_score_, 2))

Mejores parámetros:
{'regressor__max_depth': 10, 'regressor__min_samples_leaf': 2, 'regressor__min_samples_split': 2, 'regressor__n_estimators': 200}

Mejor RMSE de validación cruzada: 159296.17


In [27]:
# Evaluar el mejor modelo del Experimento 5

best_model_exp5 = grid_search.best_estimator_

pred_exp5 = best_model_exp5.predict(X_test_exp4)

mae_exp5 = mean_absolute_error(y_test_exp4, pred_exp5)
rmse_exp5 = mean_squared_error(y_test_exp4, pred_exp5) ** 0.5
r2_exp5 = r2_score(y_test_exp4, pred_exp5)

print(f'MAE Experimento 5:  ${mae_exp5:,.2f} MXN')
print(f'RMSE Experimento 5: ${rmse_exp5:,.2f} MXN')
print(f'R² Experimento 5:   {r2_exp5:.4f}')

MAE Experimento 5:  $107,411.20 MXN
RMSE Experimento 5: $152,010.18 MXN
R² Experimento 5:   0.6309


Resultados del Experimento 5

El ajuste de hiperparámetros permitió mejorar el desempeño del modelo Random Forest. El MAE disminuyó de 108,486.47 a 107,411.20 MXN, mientras que el RMSE disminuyó de 158,551.03 a 152,010.18 MXN. El R² aumentó de 0.5984 a 0.6309.

La configuración seleccionada mediante validación cruzada fue:

n_estimators = 200
max_depth = 10
min_samples_split = 2
min_samples_leaf = 2

El ajuste permitió obtener un modelo ligeramente más preciso y con una mejor capacidad para explicar la variabilidad de los precios.

Conclusión: el ajuste de hiperparámetros mejoró el desempeño de Random Forest, convirtiéndose en el mejor modelo de los experimentos realizados.

## 7) Cuando el target deja de ser numerico

Contraste rapido con el mismo dataset, ahora prediciendo categorias.

**Regresion Logistica** — ¿el cliente comprara el vehiculo? Target 0/1,
salida `P(compra=1)`. Se evalua con Accuracy, Precision, Recall y F1
(Precision/Recall son especialmente importantes si las clases estan
desbalanceadas).

**KNN** — ¿a que categoria pertenece este vehiculo (Sedan/SUV)? Busca las
`K` observaciones mas cercanas y vota. **KNN depende de distancias**: si
`km` esta en una escala mucho mayor que `anio`, dominara la distancia. Por
eso el escalamiento es especialmente importante para KNN — no existe un
preprocesamiento identico para todos los algoritmos.

In [30]:
# Regresion Logistica: ¿el cliente comprara el vehiculo?
rng = np.random.default_rng(7)
logit = -1.0 + 0.28 * (df['anio'] - 2015) - 0.00003 * df['km'] + rng.normal(0, 0.6, size=len(df))
prob_compra = 1 / (1 + np.exp(-logit))
df['compra_probable'] = rng.binomial(1, prob_compra)

X_c = df[['anio', 'km', 'transmision']]
y_c = df['compra_probable']
X_c_train, X_c_test, y_c_train, y_c_test = train_test_split(
    X_c, y_c, test_size=0.2, random_state=42, stratify=y_c
)

logit_model = Pipeline(steps=[
    ('preprocessor', ColumnTransformer(transformers=[
        ('transmision', OneHotEncoder(handle_unknown='ignore'), ['transmision']),
    ], remainder='passthrough')),
    ('classifier', LogisticRegression(max_iter=1000)),
])
logit_model.fit(X_c_train, y_c_train)
pred_c = logit_model.predict(X_c_test)

print('Accuracy: ', round(accuracy_score(y_c_test, pred_c), 4))
print('Precision:', round(precision_score(y_c_test, pred_c, zero_division=0), 4))
print('Recall:   ', round(recall_score(y_c_test, pred_c, zero_division=0), 4))
print('F1:       ', round(f1_score(y_c_test, pred_c, zero_division=0), 4))

Accuracy:  0.8998
Precision: 0.0
Recall:    0.0
F1:        0.0


In [31]:
# KNN: ¿es una "buena compra"? (anio reciente y km bajo) — sensible a escala
df['buena_compra'] = ((df['anio'] >= 2020) & (df['km'] <= 60000)).astype(int)
print(df['buena_compra'].value_counts())
print()

X_k = df[['anio', 'km']]
y_k = df['buena_compra']
X_k_train, X_k_test, y_k_train, y_k_test = train_test_split(
    X_k, y_k, test_size=0.2, random_state=42, stratify=y_k
)

knn_raw = KNeighborsClassifier(n_neighbors=5)
knn_raw.fit(X_k_train, y_k_train)
acc_raw = accuracy_score(y_k_test, knn_raw.predict(X_k_test))

scaler = StandardScaler()
X_k_train_scaled = scaler.fit_transform(X_k_train)
X_k_test_scaled = scaler.transform(X_k_test)

knn_scaled = KNeighborsClassifier(n_neighbors=5)
knn_scaled.fit(X_k_train_scaled, y_k_train)
acc_scaled = accuracy_score(y_k_test, knn_scaled.predict(X_k_test_scaled))

print(f'KNN sin escalar -> accuracy: {acc_raw:.4f}')
print(f'KNN escalado    -> accuracy: {acc_scaled:.4f}')
print()
print('anio va de 2015 a 2023 (rango ~8). km llega a ~150,000 (rango mucho mayor).')
print('Sin escalar, la distancia la domina km; anio casi no influye en el vecino mas cercano.')

buena_compra
0    4294
1      46
Name: count, dtype: int64

KNN sin escalar -> accuracy: 0.9965
KNN escalado    -> accuracy: 0.9988

anio va de 2015 a 2023 (rango ~8). km llega a ~150,000 (rango mucho mayor).
Sin escalar, la distancia la domina km; anio casi no influye en el vecino mas cercano.


## 8) Del modelo al servicio de inferencia

Esto ocurre **fuera** de la API (entrenamiento offline):

```
Datos historicos → Preparacion → Train/Test → Entrenamiento → Evaluacion → vehicle_price_v1.joblib
```

Y esto ocurre **dentro** de la API (inferencia):

```
Usuario captura marca/modelo/anio/km/transmision
        ↓
POST /predict-price
        ↓
Validacion (Pydantic)
        ↓
Carga pipeline/modelo (joblib)
        ↓
predict()
        ↓
{"estimated_price": 318500, "currency": "MXN", "model_version": "v1"}
        ↓
Frontend
```

**Punto central: entrenar y predecir son procesos diferentes.** La API usa
un modelo ya entrenado; nunca reentrena en cada request.

Guardamos el pipeline que entrenamos arriba para que el servicio de
`proyecto_web_precio_vehiculos/` lo use directamente.

In [29]:
# Guardar el mejor modelo y sus métricas

project_root = Path('proyecto_web_precio_vehiculos')

models_dir = project_root / 'models'

models_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(
    best_model_exp5,
    models_dir / 'vehicle_price_v1.joblib'
)

metrics = {
    'mae': round(float(mae_exp5), 2),
    'rmse': round(float(rmse_exp5), 2),
    'r2': round(float(r2_exp5), 4),
    'features': FEATURE_ORDER_EXP4,
    'target': TARGET,
    'model_type': 'RandomForestRegressor',
    'model_version': 'v1'
}

with open(models_dir / 'metrics.json', 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2)

print('Mejor modelo y métricas guardados en proyecto_web_precio_vehiculos/models')

Mejor modelo y métricas guardados en proyecto_web_precio_vehiculos/models


## 9) Arquitectura minima de una solucion de ML

```
FUENTES (BD, CSV/Parquet, APIs)
        ↓
DATA / TRAINING
  Validacion → Preparacion → Entrenamiento → Evaluacion → Modelo versionado
        ↓
INFERENCE SERVICE (FastAPI /predict-price)
        ↓
Pipeline (Preprocessing + Modelo)
        ↓
CONSUMIDORES (Frontend, otra app, otro sistema)
```

Bloques transversales:

- **Seguridad**: autenticacion (quien consume), validacion (que datos
  aceptamos — ya lo hace Pydantic), secrets fuera del codigo, HTTPS en
  transito.
- **Buenas practicas**: versionar el modelo (`vehicle_price_v1`), logging,
  testing (preprocessing, modelo y endpoint) y monitoring (disponibilidad +
  comportamiento del modelo/datos).

Todo esto ya esta implementado, con el mismo caso de vehiculos, en
`proyecto_web_precio_vehiculos/`. Para correrlo:

```bash
cd proyecto_web_precio_vehiculos
pip install -r requirements.txt
python scripts/train_model.py   # reentrena si hace falta
bash scripts/run_site.sh        # frontend :9010, backend :9011
```

**10) Arquitectura minima de una solucion de ML**

Después de realizar los cinco experimentos, se seleccionó como mejor modelo el Random Forest optimizado del Experimento 5.

El modelo obtuvo un MAE de 107,411.20 MXN, un RMSE de 152,010.18 MXN y un R² de 0.6309.

A continuación, se utilizará este modelo para estimar el precio de tres vehículos con diferentes características.

In [28]:
# Predicción del precio de 3 vehículos

vehiculos = pd.DataFrame([
    {
        'fuel': 'Petrol',
        'seller_type': 'Individual',
        'transmision': 'Manual',
        'owner': 'First Owner',
        'anio': 2022,
        'km': 30000
    },
    {
        'fuel': 'Diesel',
        'seller_type': 'Dealer',
        'transmision': 'Automatic',
        'owner': 'First Owner',
        'anio': 2019,
        'km': 60000
    },
    {
        'fuel': 'Petrol',
        'seller_type': 'Individual',
        'transmision': 'Manual',
        'owner': 'Second Owner',
        'anio': 2015,
        'km': 120000
    }
])

predicciones = best_model_exp5.predict(vehiculos)

resultados_prediccion = vehiculos.copy()
resultados_prediccion['precio_estimado'] = predicciones.round(2)

display(resultados_prediccion)

,fuel,seller_type,transmision,owner,anio,km,precio_estimado
0,Petrol,Individual,Manual,First Owner,2022,30000,369640.35
1,Diesel,Dealer,Automatic,First Owner,2019,60000,907320.00
2,Petrol,Individual,Manual,Second Owner,2015,120000,282277.01


Interpretación de las predicciones

Las predicciones muestran cómo el modelo estima diferentes precios dependiendo de las características de cada vehículo.

El primer vehículo, con año 2022 y 30,000 km, obtuvo un precio estimado de 369,640.35 MXN. El segundo vehículo, un modelo Diesel con transmisión automática y vendido por un Dealer, obtuvo la estimación más alta, de 907,320.00 MXN. Finalmente, el tercer vehículo, con mayor antigüedad y 120,000 km, obtuvo un precio estimado de 282,277.01 MXN.

Estas predicciones son estimaciones realizadas por el modelo y no representan necesariamente el precio real de mercado. Su precisión está limitada por las variables disponibles en el dataset y por el desempeño obtenido durante la evaluación.

11) Comparación final de experimentos

Se compararon los resultados obtenidos en el Baseline y en los cinco experimentos realizados. Para evaluar el desempeño de los modelos se utilizaron las métricas MAE, RMSE y R².

En MAE y RMSE, valores menores representan un mejor desempeño, mientras que en R² valores mayores indican que el modelo explica una mayor proporción de la variabilidad de los precios.

In [34]:
# Tabla comparativa de experimentos

resultados_experimentos = pd.DataFrame([
    {
        'Modelo': 'Baseline - Regresión Lineal',
        'MAE': mae,
        'RMSE': rmse,
        'R²': r2
    },
    {
        'Modelo': 'Experimento 1 - Antigüedad',
        'MAE': mae_exp1,
        'RMSE': rmse_exp1,
        'R²': r2_exp1
    },
    {
        'Modelo': 'Experimento 2 - Tratamiento de outliers',
        'MAE': mae_exp2,
        'RMSE': rmse_exp2,
        'R²': r2_exp2
    },
    {
        'Modelo': 'Experimento 3 - Incorporación de name',
        'MAE': mae_exp3,
        'RMSE': rmse_exp3,
        'R²': r2_exp3
    },
    {
        'Modelo': 'Experimento 4 - Random Forest',
        'MAE': mae_exp4,
        'RMSE': rmse_exp4,
        'R²': r2_exp4
    },
    {
        'Modelo': 'Experimento 5 - Random Forest optimizado',
        'MAE': mae_exp5,
        'RMSE': rmse_exp5,
        'R²': r2_exp5
    }
])

# Formato de la tabla
resultados_experimentos.style.format({
    'MAE': '${:,.0f}',
    'RMSE': '${:,.0f}',
    'R²': '{:.4f}'
})

,Modelo,MAE,RMSE,R²
0,Baseline - Regresión Lineal,"$221,706","$426,787",0.4031
1,Experimento 1 - Antigüedad,"$221,706","$426,787",0.4031
2,Experimento 2 - Tratamiento de outliers,"$125,693","$168,867",0.5445
3,Experimento 3 - Incorporación de name,"$128,556","$171,486",0.5302
4,Experimento 4 - Random Forest,"$108,486","$158,551",0.5984
5,Experimento 5 - Random Forest optimizado,"$107,411","$152,010",0.6309


12) Mejor modelo y mejora respecto al Baseline

El mejor desempeño fue obtenido por el Random Forest optimizado del Experimento 5.

El modelo final obtuvo:

MAE: 107,411.20 MXN
RMSE: 152,010.18 MXN
R²: 0.6309

Comparado con el Baseline, que obtuvo un MAE de 221,706.37 MXN, el modelo final redujo el error absoluto medio aproximadamente un 51.56%.

El RMSE disminuyó de 426,786.69 MXN a 152,010.18 MXN, representando una mejora aproximada del 64.37%.

El R² aumentó de 0.4031 a 0.6309, lo que representa un incremento de 22.78 puntos porcentuales.

In [33]:
# Mejora del mejor modelo respecto al Baseline

mejora_mae = (mae - mae_exp5) / mae * 100
mejora_rmse = (rmse - rmse_exp5) / rmse * 100
mejora_r2 = (r2_exp5 - r2) * 100

print(f'Mejora del MAE:  {mejora_mae:.2f}%')
print(f'Mejora del RMSE: {mejora_rmse:.2f}%')
print(f'Mejora del R²:   {mejora_r2:.2f} puntos porcentuales')

Mejora del MAE:  51.55%
Mejora del RMSE: 64.38%
Mejora del R²:   22.77 puntos porcentuales


13) Conclusiones

Se realizaron cinco experimentos con el objetivo de mejorar el modelo de predicción de precios de vehículos.

El modelo inicial de Regresión Lineal obtuvo un MAE de 221,706.37 MXN, un RMSE de 426,786.69 MXN y un R² de 0.4031.

En el Experimento 1 se agregó la variable antiguedad. Esta modificación no produjo cambios en las métricas, debido a que la antigüedad es una transformación directa del año del vehículo y no aportó información adicional al modelo.

En el Experimento 2 se realizó un tratamiento de valores atípicos utilizando el rango intercuartílico (IQR). Esta modificación produjo la mayor mejora individual, reduciendo considerablemente los errores y aumentando el R² de 0.4031 a 0.5445.

En el Experimento 3 se incorporó la variable name como característica categórica. El desempeño empeoró ligeramente, posiblemente debido a la gran cantidad de categorías diferentes que se generaron mediante One-Hot Encoding.

En el Experimento 4 se cambió el algoritmo de Regresión Lineal a Random Forest. Este modelo permitió capturar relaciones no lineales entre las características de los vehículos y su precio, aumentando el R² hasta 0.5984.

Finalmente, en el Experimento 5 se realizó un ajuste de hiperparámetros de Random Forest mediante GridSearchCV. La configuración seleccionada utilizó 200 árboles, una profundidad máxima de 10, min_samples_split=2 y min_samples_leaf=2.

Este modelo obtuvo un MAE de 107,411.20 MXN, un RMSE de 152,010.18 MXN y un R² de 0.6309, convirtiéndose en el mejor modelo de los experimentos realizados.

En comparación con el Baseline, el modelo final redujo el MAE en aproximadamente 51.56% y el RMSE en 64.37%, mientras que el R² aumentó de 0.4031 a 0.6309.

Por lo tanto, el Random Forest optimizado del Experimento 5 fue seleccionado como el modelo final y posteriormente se utilizó para estimar el precio de tres vehículos con diferentes características.